This script loads the eval set and trained model from the gc bucket, do inference and store the results in inference_runs folder in timestamped fashion.

In [32]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [33]:
import pandas as pd
import joblib
from google.cloud import storage
from datetime import datetime
from sklearn.metrics import accuracy_score

#Configuration paths
BUCKET_URI = f"gs://mlops-assignment-mlops-iitmadras-trialrun"
TRAIN_RUNS_URI = f"{BUCKET_URI}/train_runs/"
INFERENCE_RUNS_URI = f"{BUCKET_URI}/inference_runs/"
DATASET_FOLDER_URI = f"{BUCKET_URI}/data/raw/"

### Fetching the latest model

In [34]:
!mkdir latest_fetched_model

mkdir: cannot create directory ‘latest_fetched_model’: File exists


In [35]:
latest_runs = !gsutil ls $TRAIN_RUNS_URI | sort | tail -n 1
latest_run = latest_runs[0]
print(f"found the latest trained model at {latest_run}")

found the latest trained model at gs://mlops-assignment-mlops-iitmadras-trialrun/train_runs/20260217-0731/


In [36]:
#Copying the latest model to the instance storage
!gsutil cp {latest_run}model.joblib latest_fetched_model/

Copying gs://mlops-assignment-mlops-iitmadras-trialrun/train_runs/20260217-0731/model.joblib...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      


### Performing Inference

In [37]:
import pandas as pd
import joblib
from datetime import datetime
from sklearn.metrics import accuracy_score

timestamp = datetime.now().strftime("%Y%m%d-%H%M")
output_uri = f"{INFERENCE_RUNS_URI}{timestamp}"

# Load Data
df_eval = pd.read_csv(f"{DATASET_FOLDER_URI}eval_set.csv")
model = joblib.load("latest_fetched_model/model.joblib")

# Perform Inference
y_true = df_eval['species']
X_eval = df_eval.drop('species', axis=1)

y_pred = model.predict(X_eval)
acc = accuracy_score(y_true, y_pred)
print(f"The accuracy score for this run is {acc}")

The accuracy score for this run is 0.8888888888888888


### Saving the output files and log message to the Inference Runs folder in GS Bucket

In [39]:
# Save Artifacts using Pandas to_csv (supports gs:// natively)
# Save Predictions
pd.DataFrame({'actual': y_true, 'predicted': y_pred}).to_csv(f"{output_uri}/actual_and_predictions.csv", index=False)

# Save Log
log_msg = f"Inference Accuracy: {acc} for the inference run at {timestamp} using the model at {latest_run}"
!echo "{log_msg}" > inference_log.txt
!gsutil cp inference_log.txt {output_uri}/inference_log.txt

print(f"Done! Results are in {output_uri}")

Copying file://inference_log.txt [Content-Type=text/plain]...
/ [1 files][  170.0 B/  170.0 B]                                                
Operation completed over 1 objects/170.0 B.                                      
Done! Results are in gs://mlops-assignment-mlops-iitmadras-trialrun/inference_runs/20260217-0733
